In [5]:
import pandas as pd
import os
import numpy as np

In [7]:


# =====================================================
# LOAD DATA
# =====================================================

nav = pd.read_csv(
    r"data/raw/downloaded files/02_nav_history.csv"
)

# =====================================================
# STANDARDIZE COLUMN NAMES
# =====================================================

nav.columns = (
    nav.columns
    .str.lower()
    .str.strip()
)

print("Columns:")
print(nav.columns.tolist())

print("\nOriginal Shape:")
print(nav.shape)

# =====================================================
# CHECK MISSING VALUES
# =====================================================

print("\nMissing Values:")
print(nav.isnull().sum())

# =====================================================
# CONVERT DATE TO DATETIME
# =====================================================

nav["date"] = pd.to_datetime(
    nav["date"],
    errors="coerce"
)

# =====================================================
# CONVERT NAV TO NUMERIC
# =====================================================

nav["nav"] = pd.to_numeric(
    nav["nav"],
    errors="coerce"
)

# =====================================================
# SORT BY AMFI CODE AND DATE
# =====================================================

nav = nav.sort_values(
    ["amfi_code", "date"]
)

# =====================================================
# FORWARD FILL MISSING NAV VALUES
# =====================================================

nav["nav"] = (
    nav.groupby("amfi_code")["nav"]
    .ffill()
)

# =====================================================
# REMOVE DUPLICATES
# =====================================================

duplicates = nav.duplicated(
    subset=["amfi_code", "date"]
).sum()

print(
    "\nDuplicate Rows Found:",
    duplicates
)

nav = nav.drop_duplicates(
    subset=["amfi_code", "date"]
)

# =====================================================
# VALIDATE NAV > 0
# =====================================================

invalid_nav = nav[
    nav["nav"] <= 0
]

print(
    "\nInvalid NAV Records:",
    len(invalid_nav)
)

nav = nav[
    nav["nav"] > 0
]

# =====================================================
# FINAL DATA QUALITY SUMMARY
# =====================================================

print("\nFinal Shape:")
print(nav.shape)

print("\nMissing Values After Cleaning:")
print(nav.isnull().sum())

print("\nDate Range:")
print("Start:", nav["date"].min())
print("End  :", nav["date"].max())

# =====================================================
# SAVE CLEANED FILE
# =====================================================

output_file = (
    r"data/processed/02_nav_history_cleaned.csv"
)

nav.to_csv(
    output_file,
    index=False
)

print(
    f"\nCleaned file saved successfully:\n{output_file}"
)

Columns:
['amfi_code', 'date', 'nav']

Original Shape:
(46000, 3)

Missing Values:
amfi_code    0
date         0
nav          0
dtype: int64

Duplicate Rows Found: 0

Invalid NAV Records: 0

Final Shape:
(46000, 3)

Missing Values After Cleaning:
amfi_code    0
date         0
nav          0
dtype: int64

Date Range:
Start: 2022-01-03 00:00:00
End  : 2026-05-29 00:00:00

Cleaned file saved successfully:
data/processed/02_nav_history_cleaned.csv


In [4]:
import pandas as pd
import numpy as np

# =====================================================
# LOAD DATA
# =====================================================

transactions = pd.read_csv(
    r"data/raw/downloaded files/08_investor_transactions.csv"
)

# =====================================================
# STANDARDIZE COLUMN NAMES
# =====================================================

transactions.columns = (
    transactions.columns
    .str.lower()
    .str.strip()
)

print("Columns:")
print(transactions.columns.tolist())

print("\nOriginal Shape:")
print(transactions.shape)

# =====================================================
# CHECK MISSING VALUES
# =====================================================

print("\nMissing Values:")
print(transactions.isnull().sum())

# =====================================================
# FIX DATE FORMAT
# =====================================================

transactions["transaction_date"] = pd.to_datetime(
    transactions["transaction_date"],
    errors="coerce"
)

# =====================================================
# STANDARDIZE TRANSACTION TYPES
# =====================================================

transactions["transaction_type"] = (
    transactions["transaction_type"]
    .astype(str)
    .str.strip()
    .str.lower()
)

mapping = {
    "sip": "SIP",
    "systematic investment plan": "SIP",
    "lumpsum": "Lumpsum",
    "lump sum": "Lumpsum",
    "redemption": "Redemption",
    "redeem": "Redemption"
}

transactions["transaction_type"] = (
    transactions["transaction_type"]
    .replace(mapping)
)

print("\nTransaction Types:")
print(
    transactions["transaction_type"]
    .value_counts(dropna=False)
)

# =====================================================
# VALIDATE AMOUNT > 0
# =====================================================

transactions["amount_inr"] = pd.to_numeric(
    transactions["amount_inr"],
    errors="coerce"
)

invalid_amount = transactions[
    transactions["amount_inr"] <= 0
]

print(
    "\nInvalid Amount Records:",
    len(invalid_amount)
)

transactions = transactions[
    transactions["amount_inr"] > 0
]

# =====================================================
# CHECK KYC ENUM VALUES
# =====================================================

print("\nKYC Status Values:")
print(
    transactions["kyc_status"]
    .value_counts(dropna=False)
)

valid_kyc = [
    "Verified",
    "Pending",
    "Rejected"
]

invalid_kyc = transactions[
    ~transactions["kyc_status"].isin(valid_kyc)
]

print(
    "\nInvalid KYC Records:",
    len(invalid_kyc)
)

# =====================================================
# REMOVE DUPLICATES
# =====================================================

duplicates = transactions.duplicated().sum()

print(
    "\nDuplicate Rows Found:",
    duplicates
)

transactions = transactions.drop_duplicates()

# =====================================================
# FINAL VALIDATION
# =====================================================

print("\nFinal Shape:")
print(transactions.shape)

print("\nMissing Values After Cleaning:")
print(transactions.isnull().sum())

# =====================================================
# SAVE CLEANED FILE
# =====================================================

output_path = (
    r"data/processed/08_investor_transactions_cleaned.csv"
)

transactions.to_csv(
    output_path,
    index=False
)

print(
    f"\nCleaned dataset saved to:\n{output_path}"
)

Columns:
['investor_id', 'transaction_date', 'amfi_code', 'transaction_type', 'amount_inr', 'state', 'city', 'city_tier', 'age_group', 'gender', 'annual_income_lakh', 'payment_mode', 'kyc_status']

Original Shape:
(32778, 13)

Missing Values:
investor_id           0
transaction_date      0
amfi_code             0
transaction_type      0
amount_inr            0
state                 0
city                  0
city_tier             0
age_group             0
gender                0
annual_income_lakh    0
payment_mode          0
kyc_status            0
dtype: int64

Transaction Types:
transaction_type
SIP           19716
Lumpsum        8095
Redemption     4967
Name: count, dtype: int64

Invalid Amount Records: 0

KYC Status Values:
kyc_status
Verified    30146
Pending      2632
Name: count, dtype: int64

Invalid KYC Records: 0

Duplicate Rows Found: 0

Final Shape:
(32778, 13)

Missing Values After Cleaning:
investor_id           0
transaction_date      0
amfi_code             0
transactio

In [6]:


# =====================================================
# LOAD DATA
# =====================================================

performance = pd.read_csv(
    r"data/raw/downloaded files/07_scheme_performance.csv"
)

# =====================================================
# STANDARDIZE COLUMN NAMES
# =====================================================

performance.columns = (
    performance.columns
    .str.lower()
    .str.strip()
)

print("Columns:")
print(performance.columns.tolist())

print("\nOriginal Shape:")
print(performance.shape)

# =====================================================
# CHECK MISSING VALUES
# =====================================================

print("\nMissing Values:")
print(performance.isnull().sum())

# =====================================================
# CONVERT NUMERIC COLUMNS
# =====================================================

numeric_cols = [
    "return_1yr_pct",
    "return_3yr_pct",
    "return_5yr_pct",
    "benchmark_3yr_pct",
    "alpha",
    "beta",
    "sharpe_ratio",
    "sortino_ratio",
    "std_dev_ann_pct",
    "max_drawdown_pct",
    "aum_crore",
    "expense_ratio_pct",
    "morningstar_rating"
]

for col in numeric_cols:
    performance[col] = pd.to_numeric(
        performance[col],
        errors="coerce"
    )

# =====================================================
# VALIDATE RETURNS ARE NUMERIC
# =====================================================

return_cols = [
    "return_1yr_pct",
    "return_3yr_pct",
    "return_5yr_pct"
]

print("\nReturn Column Data Types:")
print(performance[return_cols].dtypes)

# =====================================================
# FLAG RETURN ANOMALIES
# =====================================================

return_anomalies = performance[
    (
        performance[return_cols] > 200
    ).any(axis=1)
    |
    (
        performance[return_cols] < -100
    ).any(axis=1)
]

print(
    "\nReturn Anomalies Found:",
    len(return_anomalies)
)

if len(return_anomalies) > 0:
    print(
        return_anomalies[
            ["scheme_name"] + return_cols
        ].head()
    )

# =====================================================
# EXPENSE RATIO VALIDATION
# REQUIRED RANGE: 0.1% - 2.5%
# =====================================================

expense_anomalies = performance[
    (
        performance["expense_ratio_pct"] < 0.1
    )
    |
    (
        performance["expense_ratio_pct"] > 2.5
    )
]

print(
    "\nExpense Ratio Anomalies:",
    len(expense_anomalies)
)

if len(expense_anomalies) > 0:
    print(
        expense_anomalies[
            [
                "scheme_name",
                "expense_ratio_pct"
            ]
        ].head()
    )

# =====================================================
# OPTIONAL QUALITY CHECKS
# =====================================================

negative_aum = performance[
    performance["aum_crore"] <= 0
]

print(
    "\nInvalid AUM Records:",
    len(negative_aum)
)

invalid_rating = performance[
    (
        performance["morningstar_rating"] < 1
    )
    |
    (
        performance["morningstar_rating"] > 5
    )
]

print(
    "Invalid Morningstar Ratings:",
    len(invalid_rating)
)

# =====================================================
# DUPLICATE CHECK
# =====================================================

duplicates = performance.duplicated().sum()

print(
    "\nDuplicate Rows:",
    duplicates
)

performance = performance.drop_duplicates()

# =====================================================
# FINAL SUMMARY
# =====================================================

print("\nFINAL DATA QUALITY SUMMARY")
print("-" * 40)

print("Total Records:", len(performance))
print("Return Anomalies:", len(return_anomalies))
print("Expense Ratio Anomalies:", len(expense_anomalies))
print("Invalid AUM:", len(negative_aum))
print("Invalid Ratings:", len(invalid_rating))

# =====================================================
# SAVE CLEANED DATASET
# =====================================================

output_file = (
    r"data/processed/07_scheme_performance_cleaned.csv"
)

performance.to_csv(
    output_file,
    index=False
)

print(
    f"\nCleaned file saved to:\n{output_file}"
)

Columns:
['amfi_code', 'scheme_name', 'fund_house', 'category', 'plan', 'return_1yr_pct', 'return_3yr_pct', 'return_5yr_pct', 'benchmark_3yr_pct', 'alpha', 'beta', 'sharpe_ratio', 'sortino_ratio', 'std_dev_ann_pct', 'max_drawdown_pct', 'aum_crore', 'expense_ratio_pct', 'morningstar_rating', 'risk_grade']

Original Shape:
(40, 19)

Missing Values:
amfi_code             0
scheme_name           0
fund_house            0
category              0
plan                  0
return_1yr_pct        0
return_3yr_pct        0
return_5yr_pct        0
benchmark_3yr_pct     0
alpha                 0
beta                  0
sharpe_ratio          0
sortino_ratio         0
std_dev_ann_pct       0
max_drawdown_pct      0
aum_crore             0
expense_ratio_pct     0
morningstar_rating    0
risk_grade            0
dtype: int64

Return Column Data Types:
return_1yr_pct    float64
return_3yr_pct    float64
return_5yr_pct    float64
dtype: object

Return Anomalies Found: 0

Expense Ratio Anomalies: 0

Invalid